# BumbaLM - Baseline BM25 no teste (rodada TJRS 100 mil)

Avalia o BM25 **na mesma prova** dos modelos neurais (`scripts/05_evaluate.py`):

- **Consultas:** a ementa de cada processo do teste, **sem** a instrução "Instruct: ..." (ela só serve para o modelo neural).
- **Acervo:** os chunks limpos do teste. Chunks com `contem_vazamento_ementa = true` ficam de fora.
- **Relevantes:** os chunks limpos do próprio processo **e** dos processos do mesmo `grupo_duplicata` (os "gêmeos").
- **Métricas:** as mesmas fórmulas do `InformationRetrievalEvaluator` (MRR@10, nDCG@10, Acc@1/5/10, MAP@100, P@10, R@10).

**BM25 (fórmula do Lucene/Pyserini):** k1 = 0,9 e b = 0,4, os padrões do BEIR, **sem ajuste no teste**.
O texto passa por minúsculas, remoção de stopwords do português (NLTK) e stemming Snowball em português (PyStemmer).

**Saídas** (em `PASTA_SAIDA`):
- `por_consulta_test_bm25.jsonl`: uma linha por ementa, no mesmo formato dos modelos neurais (pareável pelo `query_id`);
- `metricas_test_bm25.json`: médias gerais e dos subconjuntos com e sem gêmeos;
- `relatorio_test_bm25.txt`: resumo legível.

Roda em **CPU** (não precisa de GPU) e leva poucos minutos (cerca de 1 minuto num notebook comum).

## 1. Configuração
Ajuste `PASTA_DADOS` para a pasta do Drive com `test_processos.jsonl` e `test_corpus_chunks.jsonl`.

In [ ]:
PASTA_DADOS = "/content/drive/MyDrive/BumbaLM_teste100k"
PASTA_SAIDA = PASTA_DADOS + "/avaliacao_bm25"
SPLIT = "test"

K1 = 0.9      # saturação da frequência do termo
B = 0.4       # normalização pelo tamanho do chunk
TOP_K = 100   # resultados guardados por consulta (o mesmo do avaliador neural, por causa do MAP@100)
LOTE = 256    # consultas por lote na busca

## 2. Drive e dependências

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install PyStemmer nltk

## 3. Leitura dos dados e montagem da prova
A mesma lógica da função `montar_estruturas_ir` do projeto. Os números impressos aqui precisam bater com os
que o `05_evaluate.py` imprime no node2 ("queries" e "chunks no corpus").

In [ ]:
import json, math, os, re, time, random, platform
from array import array
from collections import Counter, defaultdict

import numpy as np
import scipy
import scipy.sparse as sp


def ler_jsonl(caminho):
    with open(caminho, encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                yield json.loads(linha)


arq_processos = os.path.join(PASTA_DADOS, f"{SPLIT}_processos.jsonl")
arq_chunks = os.path.join(PASTA_DADOS, f"{SPLIT}_corpus_chunks.jsonl")
os.makedirs(PASTA_SAIDA, exist_ok=True)

processos = []
for p in ler_jsonl(arq_processos):
    pid = p["id_global"]
    processos.append({
        "id": pid,
        "ementa": str(p.get("str_ementa") or "").strip(),
        "grupo": str(p.get("grupo_duplicata") or "").strip() or pid,
    })
ids_processos = {p["id"] for p in processos}

corpus_ids, corpus_textos = [], []
chunks_do_processo = defaultdict(set)
chunks_total = 0
chunks_vazamento = 0
chunks_orfaos = 0
for c in ler_jsonl(arq_chunks):
    chunks_total += 1
    if bool(c.get("contem_vazamento_ementa", c.get("is_ementa_leak", False))):
        chunks_vazamento += 1
        continue
    if c["id_global_processo"] not in ids_processos:
        chunks_orfaos += 1
    corpus_ids.append(c["chunk_id"])
    corpus_textos.append(c.get("texto", ""))
    chunks_do_processo[c["id_global_processo"]].add(c["chunk_id"])

# Chunks limpos de cada grupo de quase duplicatas
chunks_do_grupo = defaultdict(set)
for p in processos:
    chunks_do_grupo[p["grupo"]] |= chunks_do_processo.get(p["id"], set())

consultas = []
processos_sem_chunk_limpo = 0
for p in processos:
    proprios = chunks_do_processo.get(p["id"], set())
    if not proprios:
        processos_sem_chunk_limpo += 1
        continue
    if not p["ementa"]:
        continue
    relevantes = chunks_do_grupo[p["grupo"]]
    consultas.append({
        "query_id": f"query_{p['id']}",
        "ementa": p["ementa"],
        "relevantes": set(relevantes),
        "n_proprios": len(proprios),
        "tem_gemeos": len(relevantes) > len(proprios),
    })

protocolo = {
    "processos_avaliados": len(processos),
    "queries": len(consultas),
    "chunks_total": chunks_total,
    "chunks_no_corpus": len(corpus_ids),
    "chunks_descartados_vazamento": chunks_vazamento,
    "processos_sem_chunk_limpo": processos_sem_chunk_limpo,
    "queries_com_quase_duplicatas_relevantes": sum(c["tem_gemeos"] for c in consultas),
    "chunks_sem_processo_no_split": chunks_orfaos,
}
for chave, valor in protocolo.items():
    print(f"{chave:<42}: {valor:,}")
assert chunks_orfaos == 0, "Ha chunks de processos que nao estao no arquivo de processos. Confira os arquivos."

## 4. Tokenização (minúsculas, stopwords e stemming em português)

In [ ]:
import nltk
import Stemmer

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("portuguese"))
stemmer = Stemmer.Stemmer("portuguese")
_padrao = re.compile(r"\w+", re.UNICODE)
_cache_stem = {}


def tokenizar(texto):
    saida = []
    for palavra in _padrao.findall(texto.lower()):
        if len(palavra) < 2 or palavra in STOPWORDS:
            continue
        raiz = _cache_stem.get(palavra)
        if raiz is None:
            raiz = stemmer.stemWord(palavra)
            _cache_stem[palavra] = raiz
        saida.append(raiz)
    return saida


print(tokenizar("APELAÇÃO CÍVEL. Juros remuneratórios. Abusividade não configurada."))

## 5. Índice BM25
Peso de cada termo em cada chunk (Lucene): `idf * tf / (tf + k1 * (1 - b + b * tamanho / tamanho_medio))`,
com `idf = ln(1 + (N - df + 0,5) / (df + 0,5))`. A nota de um chunk é a soma dos pesos dos termos da consulta.

In [ ]:
t0 = time.time()
vocab = {}
linhas, colunas, valores = array("i"), array("i"), array("f")
comprimentos = np.zeros(len(corpus_textos), dtype=np.float32)
for d, texto in enumerate(corpus_textos):
    tokens = tokenizar(texto)
    comprimentos[d] = len(tokens)
    for termo, tf in Counter(tokens).items():
        t = vocab.get(termo)
        if t is None:
            t = len(vocab)
            vocab[termo] = t
        linhas.append(d)
        colunas.append(t)
        valores.append(tf)
    if (d + 1) % 20000 == 0:
        print(f"  {d + 1:,} chunks tokenizados ({time.time() - t0:.0f}s)")

N, V = len(corpus_textos), len(vocab)
linhas = np.frombuffer(linhas, dtype=np.int32)
colunas = np.frombuffer(colunas, dtype=np.int32)
valores = np.frombuffer(valores, dtype=np.float32)

df = np.bincount(colunas, minlength=V).astype(np.float64)
idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
tamanho_medio = float(comprimentos.mean())
normalizacao = K1 * (1.0 - B + B * comprimentos[linhas] / tamanho_medio)
pesos = idf[colunas] * valores / (valores + normalizacao)

# Matriz termo x chunk, pronta para multiplicar pelas consultas
W_T = sp.csr_matrix((pesos.astype(np.float32), (colunas, linhas)), shape=(V, N))
del linhas, colunas, valores, normalizacao, pesos
print(f"Indice pronto: {N:,} chunks | {V:,} termos | tamanho medio {tamanho_medio:.0f} termos | {time.time() - t0:.0f}s")

## 6. Busca

In [ ]:
def montar_matriz_consultas(textos):
    l, c, v = array("i"), array("i"), array("f")
    for i, texto in enumerate(textos):
        for termo, n in Counter(tokenizar(texto)).items():
            t = vocab.get(termo)
            if t is not None:
                l.append(i)
                c.append(t)
                v.append(n)
    return sp.csr_matrix(
        (np.frombuffer(v, dtype=np.float32), (np.frombuffer(l, dtype=np.int32), np.frombuffer(c, dtype=np.int32))),
        shape=(len(textos), V),
    )


def buscar(Q, top_k=TOP_K, lote=LOTE, mostrar=False):
    n = Q.shape[0]
    top_ids = np.zeros((n, top_k), dtype=np.int64)
    top_notas = np.zeros((n, top_k), dtype=np.float32)
    t0 = time.time()
    for ini in range(0, n, lote):
        S = (Q[ini:ini + lote] @ W_T).toarray()
        idx = np.argpartition(-S, top_k - 1, axis=1)[:, :top_k]
        notas = np.take_along_axis(S, idx, axis=1)
        ordem = np.argsort(-notas, axis=1, kind="stable")
        top_ids[ini:ini + lote] = np.take_along_axis(idx, ordem, axis=1)
        top_notas[ini:ini + lote] = np.take_along_axis(notas, ordem, axis=1)
        if mostrar and (ini // lote) % 10 == 0:
            print(f"  {min(ini + lote, n):,}/{n:,} consultas ({time.time() - t0:.0f}s)")
    return top_ids, top_notas


# Conferencia rapida: usar o proprio texto de um chunk como consulta deve trazer esse chunk em 1o lugar
amostra = random.Random(42).sample(range(N), 20)
ids_conf, _ = buscar(montar_matriz_consultas([corpus_textos[d] for d in amostra]))
acertos_conf = sum(int(ids_conf[i, 0] == d) for i, d in enumerate(amostra))
print(f"Conferencia: {acertos_conf}/20 chunks se acharam em 1o lugar (esperado: 19 ou 20)")

t0 = time.time()
Q = montar_matriz_consultas([c["ementa"] for c in consultas])
print(f"Consultas tokenizadas: {Q.shape[0]:,} ({time.time() - t0:.0f}s)")
top_ids, top_notas = buscar(Q, mostrar=True)
print(f"Busca concluida em {time.time() - t0:.0f}s")

## 7. Métricas por consulta e médias
As mesmas fórmulas de `calcular_metricas_por_consulta` (em `src/evaluation/ir_evaluator.py`).

In [ ]:
def dcg(relevancias, k):
    return sum(r / math.log2(i + 2) for i, r in enumerate(relevancias[:k]))


METRICAS = ["ndcg@10", "mrr@10", "map@100", "accuracy@1", "accuracy@5", "accuracy@10", "precision@10", "recall@10"]

por_consulta = []
for i, c in enumerate(consultas):
    relevantes = c["relevantes"]
    ids = [corpus_ids[j] for j in top_ids[i]]
    acertos = [1 if x in relevantes else 0 for x in ids]
    posicao = next((k + 1 for k, a in enumerate(acertos) if a), None)

    linha = {
        "query_id": c["query_id"],
        "n_relevantes": len(relevantes),
        "posicao_primeiro_relevante": posicao,
        "tem_gemeos": c["tem_gemeos"],
    }
    for k in (1, 5, 10):
        linha[f"accuracy@{k}"] = 1.0 if any(acertos[:k]) else 0.0
    linha["mrr@10"] = 1.0 / posicao if posicao is not None and posicao <= 10 else 0.0
    linha["ndcg@10"] = dcg(acertos, 10) / dcg([1] * len(relevantes), 10)
    linha["precision@10"] = sum(acertos[:10]) / 10
    linha["recall@10"] = sum(acertos[:10]) / len(relevantes)
    soma, corretos = 0.0, 0
    for k, a in enumerate(acertos[:100]):
        if a:
            corretos += 1
            soma += corretos / (k + 1)
    linha["map@100"] = soma / min(100, len(relevantes))
    linha["top10"] = [[ids[k], round(float(top_notas[i, k]), 5)] for k in range(10)]
    por_consulta.append(linha)


def medias(linhas):
    return {m: (sum(l[m] for l in linhas) / len(linhas) if linhas else float("nan")) for m in METRICAS}


subconjuntos = {
    "todas": por_consulta,
    "sem_gemeos": [l for l in por_consulta if not l["tem_gemeos"]],
    "com_gemeos": [l for l in por_consulta if l["tem_gemeos"]],
}
resultados = {nome: {"consultas": len(ls), **medias(ls)} for nome, ls in subconjuntos.items()}

print(f"{'Metrica':<14}" + "".join(f"{n:>14}" for n in subconjuntos))
print(f"{'consultas':<14}" + "".join(f"{resultados[n]['consultas']:>14,}" for n in subconjuntos))
for m in METRICAS:
    print(f"{m:<14}" + "".join(f"{resultados[n][m]:>14.4f}" for n in subconjuntos))

## 8. Gravação no Drive

In [ ]:
caminho_consultas = os.path.join(PASTA_SAIDA, f"por_consulta_{SPLIT}_bm25.jsonl")
with open(caminho_consultas, "w", encoding="utf-8") as f:
    for linha in por_consulta:
        f.write(json.dumps(linha, ensure_ascii=False) + "\n")

parametros = {
    "k1": K1, "b": B, "top_k": TOP_K, "formula": "Lucene BM25 (idf = ln(1 + (N - df + 0.5) / (df + 0.5)))",
    "tokenizacao": "minusculas, regex \\w+, palavras com 2+ caracteres, stopwords NLTK portugues, Snowball portugues (PyStemmer)",
    "consulta": "ementa sem a instrucao 'Instruct:'",
    "vocabulario": V, "tamanho_medio_chunk_termos": tamanho_medio,
    "versoes": {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
                "nltk": nltk.__version__},
}
relatorio = {"modelo": "bm25", "split": SPLIT, "protocolo": protocolo, "parametros": parametros,
             "metricas_global": {m: resultados["todas"][m] for m in METRICAS}, "subconjuntos": resultados}
caminho_json = os.path.join(PASTA_SAIDA, f"metricas_{SPLIT}_bm25.json")
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False)

caminho_txt = os.path.join(PASTA_SAIDA, f"relatorio_{SPLIT}_bm25.txt")
with open(caminho_txt, "w", encoding="utf-8") as f:
    f.write("=" * 64 + "\nBASELINE BM25 - SPLIT " + SPLIT.upper() + "\n" + "=" * 64 + "\n\n")
    for chave, valor in protocolo.items():
        f.write(f"{chave:<42}: {valor}\n")
    f.write(f"\nk1 = {K1} | b = {B} | vocabulario = {V:,} termos\n\n")
    f.write(f"{'Metrica':<14}" + "".join(f"{n:>14}" for n in subconjuntos) + "\n")
    f.write(f"{'consultas':<14}" + "".join(f"{resultados[n]['consultas']:>14,}" for n in subconjuntos) + "\n")
    for m in METRICAS:
        f.write(f"{m:<14}" + "".join(f"{resultados[n][m]:>14.4f}" for n in subconjuntos) + "\n")

print("Arquivos gravados:")
for caminho in (caminho_consultas, caminho_json, caminho_txt):
    print(f"  {caminho} ({os.path.getsize(caminho) / 1e6:.1f} MB)")

## 9. (Opcional, depois do `avaliar` no node2) Modelos neurais com e sem gêmeos
Copie a pasta `experiments/tcc_tjrs100k_ablacao_negativos_h200/avaliacoes/test` do node2 para
`PASTA_DADOS/avaliacoes_neurais` no Drive e rode esta célula. Ela lê os arquivos `por_consulta_*.jsonl` de cada
modelo e do BM25 e monta a tabela de MRR@10 e nDCG@10 em todas as consultas, sem gêmeos e com gêmeos.

In [ ]:
import glob

PASTA_NEURAIS = PASTA_DADOS + "/avaliacoes_neurais"
gemeos = {c["query_id"]: c["tem_gemeos"] for c in consultas}

arquivos = sorted(glob.glob(os.path.join(PASTA_NEURAIS, "**", f"por_consulta_{SPLIT}_*.jsonl"), recursive=True))
arquivos.append(caminho_consultas)
print(f"{'Modelo':<14}{'consultas':>10}{'MRR todas':>12}{'MRR sem g.':>12}{'MRR com g.':>12}{'nDCG todas':>12}{'nDCG sem g.':>12}")
for arq in arquivos:
    nome = os.path.basename(arq)[len(f"por_consulta_{SPLIT}_"):-len(".jsonl")]
    linhas_modelo = list(ler_jsonl(arq))
    faltando = sum(1 for l in linhas_modelo if l["query_id"] not in gemeos)
    if faltando:
        print(f"{nome}: {faltando} consultas fora deste teste. Confira se o arquivo e da mesma rodada.")
        continue
    sem = [l for l in linhas_modelo if not gemeos[l["query_id"]]]
    com = [l for l in linhas_modelo if gemeos[l["query_id"]]]
    media = lambda ls, m: sum(l[m] for l in ls) / len(ls) if ls else float("nan")
    print(f"{nome:<14}{len(linhas_modelo):>10,}{media(linhas_modelo, 'mrr@10'):>12.4f}{media(sem, 'mrr@10'):>12.4f}"
          f"{media(com, 'mrr@10'):>12.4f}{media(linhas_modelo, 'ndcg@10'):>12.4f}{media(sem, 'ndcg@10'):>12.4f}")